# scikit-learn: The Complete Practical Guide

Notebook for **Chapter 19** of the *python* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/python/ch19.html

Tip: change a number, predict the output, then run the cell again.

## 19.1 What scikit-learn Is, When to Use It, and the Mental Model

In [ ]:
import sklearn
import numpy as np
import pandas as pd

print(sklearn.__version__)
print(np.__version__, pd.__version__)

1.9.1
2.5.3 3.0.6


## 19.2 The Estimator API, Datasets and Splitting Data

This section is the foundation. Every later section is a variation of the same four verbs: `fit`, `predict`, `transform`, `score`.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

X = np.array([[1.0], [2.0], [3.0], [4.0]])   # shape (4, 1): 4 rows, 1 feature
y = np.array([110, 190, 310, 390])

model = LinearRegression()
model.fit(X, y)
print("slope:", model.coef_, "intercept:", round(model.intercept_, 1))
print("predict size 5:", model.predict([[5.0]]))
print("R squared:", round(model.score(X, y), 4))

slope: [96.] intercept: 10.0
predict size 5: [490.]
R squared: 0.9931


In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

data = load_iris(as_frame=True)
X, y = data.data, data.target
print(X.shape, list(X.columns))

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0)
print(X_tr.shape, X_te.shape, y_te.value_counts().to_dict())

clf = LogisticRegression(max_iter=200)
clf.fit(X_tr, y_tr)
print("train acc:", round(clf.score(X_tr, y_tr), 3))
print("test acc :", round(clf.score(X_te, y_te), 3))
print("classes_:", clf.classes_, " coef_ shape:", clf.coef_.shape)
print(clf.predict_proba(X_te.iloc[:2]).round(3))

(150, 4) ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
(112, 4) (38, 4) {0: 13, 1: 13, 2: 12}
train acc: 0.955
test acc : 1.0
classes_: [0 1 2]  coef_ shape: (3, 4)
[[0.968 0.032 0.   ]
 [0.966 0.034 0.   ]]


In [ ]:
from sklearn.base import clone
from sklearn.exceptions import NotFittedError
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.validation import check_is_fitted

tree = DecisionTreeClassifier(max_depth=3, random_state=0)
print(tree.get_params()["max_depth"])

try:
    check_is_fitted(tree)
except NotFittedError:
    print("not fitted yet")

tree.set_params(max_depth=5)
copy = clone(tree)
print(copy.max_depth, copy is tree)
print(tree)

3
not fitted yet
5 False
DecisionTreeClassifier(max_depth=5, random_state=0)


In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=2000, n_features=12, n_informative=5,
                           weights=[0.9, 0.1], random_state=42)
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=42)

for name, part in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name:5s} rows={len(part):4d} positive rate={part.mean():.3f}")

train rows=1200 positive rate=0.101
val   rows= 400 positive rate=0.100
test  rows= 400 positive rate=0.100


## 19.3 Preprocessing, ColumnTransformer and Pipeline

Models want numbers without gaps, on comparable scales. Real tables have text categories, missing cells and columns measured in different units. Preprocessing closes that gap. The rule that governs this whole section: **every preprocessing step learns from the training rows only**, then applies what it learned to everything else.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

train = np.array([[10.0], [20.0], [30.0], [40.0]])
test = np.array([[100.0], [50.0]])

sc = StandardScaler()
sc.fit(train)
print("mean_:", sc.mean_, "scale_:", sc.scale_.round(3))
print(sc.transform(train).ravel().round(3))
print(sc.transform(test).ravel().round(3))

# same arithmetic by hand
print(((test - train.mean()) / train.std()).ravel().round(3))

mean_: [25.] scale_: [11.18]
[-1.342 -0.447  0.447  1.342]
[6.708 2.236]
[6.708 2.236]


In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

train = pd.DataFrame({"plan": ["basic", "plus", "basic", "pro"],
                      "size": ["S", "L", "M", "S"]})
new = pd.DataFrame({"plan": ["enterprise"], "size": ["M"]})

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe.fit(train[["plan"]])
print(ohe.get_feature_names_out())
print(ohe.transform(train[["plan"]]))
print(ohe.transform(new[["plan"]]))

ordenc = OrdinalEncoder(categories=[["S", "M", "L"]])
print(ordenc.fit_transform(train[["size"]]).ravel())

['plan_basic' 'plan_plus' 'plan_pro']
[[1. 0. 0.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]]
[[0. 0. 0.]]
[0. 2. 1. 0.]


In [ ]:
import numpy as np
from sklearn.impute import SimpleImputer

X_train = np.array([[1.0, 7.0], [2.0, np.nan], [3.0, 9.0], [np.nan, 11.0]])
X_new = np.array([[np.nan, np.nan]])

imp = SimpleImputer(strategy="median", add_indicator=True)
print(imp.fit_transform(X_train))
print(imp.statistics_)
print(imp.transform(X_new))

[[ 1.  7.  0.  0.]
 [ 2.  9.  0.  1.]
 [ 3.  9.  0.  0.]
 [ 2. 11.  1.  0.]]
[2. 9.]
[[2. 9. 1. 1.]]


In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

rng = np.random.default_rng(0)
n = 2000
df = pd.DataFrame({
    "tenure": rng.integers(1, 72, n),
    "monthly": rng.normal(65, 20, n).round(2),
    "calls": rng.poisson(2, n),
    "contract": rng.choice(["monthly", "yearly", "two-year"], n, p=[0.55, 0.25, 0.20]),
    "payment": rng.choice(["card", "bank", "cheque"], n),
})
logit = (-2.0 - 0.04 * df["tenure"] + 0.02 * df["monthly"] + 0.3 * df["calls"]
         + np.where(df["contract"] == "monthly", 1.0, 0.0))
df["churn"] = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
df.loc[rng.random(n) < 0.08, "monthly"] = np.nan

X, y = df.drop(columns="churn"), df["churn"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y, random_state=0)

num = Pipeline([("impute", SimpleImputer(strategy="median")),
                ("scale", StandardScaler())])
cat = OneHotEncoder(handle_unknown="ignore")
prep = ColumnTransformer([("num", num, ["tenure", "monthly", "calls"]),
                          ("cat", cat, ["contract", "payment"])])
pipe = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=500))])

pipe.fit(X_tr, y_tr)
print("churn rate:", round(y.mean(), 3))
print("test accuracy:", round(pipe.score(X_te, y_te), 3))
print(pipe.named_steps["prep"].get_feature_names_out())
print(pipe.named_steps["clf"].coef_.round(2))

churn rate: 0.294
test accuracy: 0.74
['num__tenure' 'num__monthly' 'num__calls' 'cat__contract_monthly'
 'cat__contract_two-year' 'cat__contract_yearly' 'cat__payment_bank'
 'cat__payment_card' 'cat__payment_cheque']
[[-0.77  0.4   0.41  0.66 -0.3  -0.36 -0.02  0.01  0.01]]


## 19.4 Model Families: Linear, Trees, Forests, Boosting, SVM, kNN, Naive Bayes, Clustering, PCA

You do not need to know all the algorithms deeply to use them well. You need to know what each assumes, what it is fast or slow at, and which settings matter. All of them share the interface from 19.2, so swapping one for another is a one-line change.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
models = {
    "dummy": DummyClassifier(strategy="prior"),
    "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "tree": DecisionTreeClassifier(max_depth=4, random_state=0),
    "forest": RandomForestClassifier(200, random_state=0),
    "hist boosting": HistGradientBoostingClassifier(random_state=0),
    "svm": make_pipeline(StandardScaler(), SVC()),
    "knn": make_pipeline(StandardScaler(), KNeighborsClassifier(15)),
    "naive bayes": GaussianNB(),
}
for name, m in models.items():
    s = cross_val_score(m, X, y, cv=cv)
    print(f"{name:14s} {s.mean():.3f} +/- {s.std():.3f}")

dummy          0.627 +/- 0.004
logistic       0.979 +/- 0.014
tree           0.926 +/- 0.025
forest         0.967 +/- 0.019
hist boosting  0.970 +/- 0.012
svm            0.977 +/- 0.007
knn            0.961 +/- 0.011
naive bayes    0.939 +/- 0.020


In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X, y = load_diabetes(return_X_y=True)
names = load_diabetes().feature_names
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0)

for label, est in [("ols", LinearRegression()), ("ridge a=10", Ridge(alpha=10)),
                   ("lasso a=2", Lasso(alpha=2.0))]:
    m = make_pipeline(StandardScaler(), est).fit(X_tr, y_tr)
    coef = m[-1].coef_
    print(f"{label:11s} R2={m.score(X_te, y_te):.3f} zeros={int((coef == 0).sum())}",
          np.round(coef[:4], 1))
print(names[:4])

ols         R2=0.359 zeros=0 [-2.1 -9.9 28.9 14.3]
ridge a=10  R2=0.358 zeros=0 [-1.7 -9.5 28.4 13.9]
lasso a=2   R2=0.355 zeros=3 [-0.  -6.4 28.8 11.5]
['age', 'sex', 'bmi', 'bp']


In [ ]:
from sklearn.datasets import load_iris, make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

iris = load_iris()
small = DecisionTreeClassifier(max_depth=2, random_state=0).fit(iris.data, iris.target)
print(export_text(small, feature_names=list(iris.feature_names)))

X, y = make_moons(600, noise=0.35, random_state=1)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=1)
for d in (1, 3, 6, None):
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_tr, y_tr)
    print(f"depth={d!s:4s} leaves={t.get_n_leaves():3d} "
          f"train={t.score(X_tr, y_tr):.3f} test={t.score(X_te, y_te):.3f}")

|--- petal width (cm) <= 0.80
|   |--- class: 0
|--- petal width (cm) >  0.80
|   |--- petal width (cm) <= 1.75
|   |   |--- class: 1
|   |--- petal width (cm) >  1.75
|   |   |--- class: 2

depth=1    leaves=  2 train=0.783 test=0.783
depth=3    leaves=  7 train=0.874 test=0.850
depth=6    leaves= 23 train=0.910 test=0.828
depth=None leaves= 68 train=1.000 test=0.822


In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split

X, y = make_classification(4000, 20, n_informative=6, random_state=0)
rng = np.random.default_rng(0)
X[rng.random(X.shape) < 0.1] = np.nan           # blank out 10% of cells
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0)

hgb = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1,
                                     early_stopping=True, random_state=0)
hgb.fit(X_tr, y_tr)
print("hist boosting test acc:", round(hgb.score(X_te, y_te), 3),
      "iterations used:", hgb.n_iter_)

rf = RandomForestClassifier(200, oob_score=True, random_state=0)
rf.fit(X_tr, y_tr)
print("forest test acc:", round(rf.score(X_te, y_te), 3))
print("forest OOB score:", round(rf.oob_score_, 3))

hist boosting test acc: 0.889 iterations used: 75
forest test acc: 0.876
forest OOB score: 0.876


In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
n = 600
signal = rng.normal(size=n)                       # small-scale informative column
y = (signal + rng.normal(scale=0.5, size=n) > 0).astype(int)
big_noise = rng.normal(scale=1000, size=n)         # huge-scale useless column
X = np.c_[signal, big_noise]

raw = KNeighborsClassifier(15)
scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(15))
print("raw   :", cross_val_score(raw, X, y, cv=5).mean().round(3))
print("scaled:", cross_val_score(scaled, X, y, cv=5).mean().round(3))

raw   : 0.518
scaled: 0.877


In [ ]:
import numpy as np
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import load_wine, make_moons
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_wine(return_X_y=True)
Xs = StandardScaler().fit_transform(X)
pca = PCA().fit(Xs)
print("explained variance ratio:", pca.explained_variance_ratio_[:4].round(3))
print("components for 90%:", int(np.searchsorted(np.cumsum(pca.explained_variance_ratio_), 0.90) + 1))

Z = PCA(n_components=2).fit_transform(Xs)
km = KMeans(3, n_init=10, random_state=0).fit(Z)
print("kmeans vs true wine class, ARI:", round(adjusted_rand_score(y, km.labels_), 3))

M, truth = make_moons(400, noise=0.07, random_state=0)
km2 = KMeans(2, n_init=10, random_state=0).fit(M)
db = DBSCAN(eps=0.2, min_samples=5).fit(M)
print("moons ARI, kmeans:", round(adjusted_rand_score(truth, km2.labels_), 3))
print("moons ARI, dbscan:", round(adjusted_rand_score(truth, db.labels_), 3),
      "clusters:", len(set(db.labels_) - {-1}), "noise:", int((db.labels_ == -1).sum()))

explained variance ratio: [0.362 0.192 0.111 0.071]
components for 90%: 8
kmeans vs true wine class, ARI: 0.895
moons ARI, kmeans: 0.269
moons ARI, dbscan: 1.0 clusters: 2 noise: 0


In [ ]:
# needs a full environment: pip install lightgbm xgboost
# not run here: needs `pip install xgboost lightgbm`
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

xgb = XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=4,
                    subsample=0.8, colsample_bytree=0.8, eval_metric="logloss")
lgbm = LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31)
# both drop straight into Pipeline, cross_val_score and GridSearchCV

## 19.5 Cross-Validation, GridSearchCV and RandomizedSearchCV

One train/test split gives one number, and one number can be lucky. **Cross-validation (CV)** splits the training data into k folds, trains k times (each time holding out a different fold), and gives you k scores. The mean tells you the typical quality, and the spread tells you how much to trust it. Hyper-parameter searches are CV run once for every candidate setting.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

print(cross_val_score(pipe, X, y, cv=5).round(3))
res = cross_validate(pipe, X, y, cv=5, scoring=["accuracy", "roc_auc", "f1"],
                     return_train_score=True)
print(sorted(res.keys()))
for k in ("test_accuracy", "train_accuracy", "test_roc_auc", "test_f1"):
    print(f"{k:15s} mean={res[k].mean():.3f}")

[0.982 0.982 0.974 0.974 0.991]
['fit_time', 'score_time', 'test_accuracy', 'test_f1', 'test_roc_auc', 'train_accuracy', 'train_f1', 'train_roc_auc']
test_accuracy   mean=0.981
train_accuracy  mean=0.989
test_roc_auc    mean=0.995
test_f1         mean=0.985


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, KFold, TimeSeriesSplit, cross_val_score

rng = np.random.default_rng(0)
n_groups, per = 60, 20
signature = rng.normal(size=(n_groups, 5))            # fixed fingerprint per customer
label_of_group = rng.integers(0, 2, n_groups)          # label unrelated to the fingerprint
groups = np.repeat(np.arange(n_groups), per)
X = signature[groups] + rng.normal(scale=0.1, size=(n_groups * per, 5))
y = label_of_group[groups]

rf = RandomForestClassifier(100, random_state=0)
random_cv = KFold(5, shuffle=True, random_state=0)
print("random KFold :", cross_val_score(rf, X, y, cv=random_cv).mean().round(3))
print("GroupKFold   :", cross_val_score(rf, X, y, cv=GroupKFold(5), groups=groups).mean().round(3))

for tr, va in TimeSeriesSplit(n_splits=3).split(np.arange(12)):
    print("train", tr.tolist(), "-> validate", va.tolist())

random KFold : 0.992
GroupKFold   : 0.515
train [0, 1, 2] -> validate [3, 4, 5]
train [0, 1, 2, 3, 4, 5] -> validate [6, 7, 8]
train [0, 1, 2, 3, 4, 5, 6, 7, 8] -> validate [9, 10, 11]


In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y, random_state=0)

pipe = Pipeline([("scale", StandardScaler()), ("svm", SVC())])
grid = {"svm__C": [0.1, 1, 10], "svm__gamma": ["scale", 0.01, 0.1]}
gs = GridSearchCV(pipe, grid, cv=5, scoring="roc_auc")
gs.fit(X_tr, y_tr)

print("best params:", gs.best_params_)
print("best CV auc:", round(gs.best_score_, 4))
top = pd.DataFrame(gs.cv_results_).sort_values("rank_test_score").head(3)
print(top[["param_svm__C", "param_svm__gamma", "mean_test_score", "std_test_score"]].round(4).to_string(index=False))
print("test auc   :", round(gs.score(X_te, y_te), 4))

best params: {'svm__C': 1, 'svm__gamma': 'scale'}
best CV auc: 0.9959
 param_svm__C param_svm__gamma  mean_test_score  std_test_score
          1.0            scale           0.9959          0.0052
          1.0             0.01           0.9959          0.0060
         10.0             0.01           0.9951          0.0077
test auc   : 0.9927


In [ ]:
from scipy.stats import loguniform, randint
from sklearn.datasets import make_classification
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split

X, y = make_classification(3000, 20, n_informative=8, flip_y=0.05, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0)

dist = {"learning_rate": loguniform(0.02, 0.3),
        "max_leaf_nodes": randint(8, 64),
        "min_samples_leaf": randint(10, 80),
        "l2_regularization": loguniform(1e-3, 10)}
rs = RandomizedSearchCV(HistGradientBoostingClassifier(max_iter=150, random_state=0),
                        dist, n_iter=12, cv=3, scoring="accuracy", random_state=0)
rs.fit(X_tr, y_tr)
print({k: (round(v, 3) if isinstance(v, float) else int(v)) for k, v in rs.best_params_.items()})
print("CV:", round(rs.best_score_, 3), " test:", round(rs.score(X_te, y_te), 3))

{'l2_regularization': np.float64(0.081), 'learning_rate': np.float64(0.18), 'max_leaf_nodes': 54, 'min_samples_leaf': 47}
CV: 0.925  test: 0.919


In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import validation_curve
from sklearn.tree import DecisionTreeClassifier

X, y = make_moons(600, noise=0.35, random_state=1)
depths = np.arange(1, 16)
tr, va = validation_curve(DecisionTreeClassifier(random_state=0), X, y,
                          param_name="max_depth", param_range=depths, cv=5)
best = depths[va.mean(axis=1).argmax()]
print("best depth:", best, " val acc:", va.mean(axis=1).max().round(3))
print("depth 15  : train", tr.mean(axis=1)[-1].round(3), " val", va.mean(axis=1)[-1].round(3))

best depth: 5  val acc: 0.86
depth 15  : train 0.997  val 0.812


## 19.6 Metrics, Class Imbalance and Calibration

A model is only "good" relative to a question. "Be right as often as possible" and "catch every fraud without drowning the analysts" are different questions with different metrics. This section gives you the metrics, then shows how imbalance breaks the default one and how to fix it.

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

y_true = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
y_pred = np.array([1, 1, 0, 0, 1, 0, 0, 0, 0, 0])

print(confusion_matrix(y_true, y_pred))
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
print("tn fp fn tp:", tn, fp, fn, tp)
print("precision:", precision_score(y_true, y_pred), "= tp/(tp+fp) =", tp / (tp + fp))
print("recall   :", recall_score(y_true, y_pred), "= tp/(tp+fn) =", tp / (tp + fn))
print("f1       :", round(f1_score(y_true, y_pred), 3))

[[5 1]
 [2 2]]
tn fp fn tp: 5 1 2 2
precision: 0.6666666666666666 = tp/(tp+fp) = 0.6666666666666666
recall   : 0.5 = tp/(tp+fn) = 0.5
f1       : 0.571


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.datasets import make_classification
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             classification_report, roc_auc_score)
from sklearn.model_selection import train_test_split

X, y = make_classification(20000, 15, n_informative=5, weights=[0.99], flip_y=0.0,
                           class_sep=0.5, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y, random_state=0)
print("positive rate:", round(y.mean(), 4))

dummy = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
model = HistGradientBoostingClassifier(random_state=0).fit(X_tr, y_tr)
for name, m in [("dummy", dummy), ("hist boosting", model)]:
    p = m.predict(X_te)
    print(f"{name:14s} acc={accuracy_score(y_te, p):.4f} "
          f"bal_acc={balanced_accuracy_score(y_te, p):.3f}")
proba = model.predict_proba(X_te)[:, 1]
print("roc_auc:", round(roc_auc_score(y_te, proba), 3),
      " avg precision:", round(average_precision_score(y_te, proba), 3))
print(classification_report(y_te, model.predict(X_te), digits=3))

positive rate: 0.01
dummy          acc=0.9900 bal_acc=0.500
hist boosting  acc=0.9922 bal_acc=0.659
roc_auc: 0.964  avg precision: 0.503
              precision    recall  f1-score   support

           0      0.993     0.999     0.996      4950
           1      0.762     0.320     0.451        50

    accuracy                          0.992      5000
   macro avg      0.878     0.659     0.723      5000
weighted avg      0.991     0.992     0.991      5000


In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve, precision_score, recall_score
from sklearn.model_selection import train_test_split

X, y = make_classification(10000, 12, n_informative=5, weights=[0.95], random_state=1)
X_tr, X_val, y_tr, y_val = train_test_split(X, y, stratify=y, random_state=1)
m = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
proba = m.predict_proba(X_val)[:, 1]

prec, rec, thr = precision_recall_curve(y_val, proba)
ok = np.where(rec[:-1] >= 0.80)[0]          # thresholds that keep recall at 80% or more
best = ok[np.argmax(prec[:-1][ok])]          # among them, the highest precision
print("chosen threshold:", round(float(thr[best]), 3))
for t in (0.5, thr[best]):
    p = (proba >= t).astype(int)
    print(f"t={t:.3f} flagged={int(p.sum()):3d} precision={precision_score(y_val, p, zero_division=0):.3f} "
          f"recall={recall_score(y_val, p):.3f}")

chosen threshold: 0.077
t=0.500 flagged=  5 precision=0.000 recall=0.000
t=0.077 flagged=522 precision=0.209 recall=0.801


In [ ]:
import numpy as np
from sklearn.metrics import (mean_absolute_error, mean_absolute_percentage_error,
                             median_absolute_error, r2_score, root_mean_squared_error)

y_true = np.array([100.0, 120, 130, 110, 125, 115])
good = np.array([102.0, 118, 133, 108, 127, 113])
one_bad = good.copy(); one_bad[0] = 160            # one big miss

for name, p in (("good", good), ("one big miss", one_bad)):
    print(f"{name:13s} MAE={mean_absolute_error(y_true, p):5.2f} "
          f"RMSE={root_mean_squared_error(y_true, p):5.2f} "
          f"MedAE={median_absolute_error(y_true, p):4.1f} "
          f"MAPE={mean_absolute_percentage_error(y_true, p):.3f} "
          f"R2={r2_score(y_true, p):6.3f}")

good          MAE= 2.17 RMSE= 2.20 MedAE= 2.0 MAPE=0.019 R2= 0.950
one big miss  MAE=11.83 RMSE=24.58 MedAE= 2.0 MAPE=0.115 R2=-5.214


In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import (adjusted_rand_score, davies_bouldin_score, ndcg_score,
                             roc_auc_score, silhouette_score)

# ranking: relevance of 5 items (3 = best) and two model score lists
rel = np.array([[3, 2, 0, 0, 1]])
good_scores = np.array([[0.9, 0.8, 0.1, 0.2, 0.5]])
bad_scores = np.array([[0.1, 0.2, 0.9, 0.8, 0.3]])
print("NDCG@3 good:", round(ndcg_score(rel, good_scores, k=3), 3),
      " bad:", round(ndcg_score(rel, bad_scores, k=3), 3))
print("AUC for binary labels:", roc_auc_score([0, 0, 1, 1], [0.1, 0.4, 0.35, 0.8]))

# clustering
X, truth = make_blobs(400, centers=3, cluster_std=1.0, random_state=0)
labels = KMeans(3, n_init=10, random_state=0).fit_predict(X)
print("silhouette:", round(silhouette_score(X, labels), 3),
      " davies-bouldin:", round(davies_bouldin_score(X, labels), 3),
      " ARI:", round(adjusted_rand_score(truth, labels), 3))

NDCG@3 good: 1.0  bad: 0.105
AUC for binary labels: 0.75
silhouette: 0.473  davies-bouldin: 0.743  ARI: 0.774


In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_classification(10000, 12, n_informative=4, weights=[0.96], flip_y=0.02, random_state=3)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y, random_state=3)

for cw in (None, "balanced"):
    m = make_pipeline(StandardScaler(), LogisticRegression(class_weight=cw, max_iter=1000))
    m.fit(X_tr, y_tr)
    p, pr = m.predict(X_te), m.predict_proba(X_te)[:, 1]
    print(f"class_weight={cw!s:9s} precision={precision_score(y_te, p):.3f} "
          f"recall={recall_score(y_te, p):.3f} AP={average_precision_score(y_te, pr):.3f}")

class_weight=None      precision=0.824 recall=0.336 AP=0.603
class_weight=balanced  precision=0.193 recall=0.808 AP=0.542


In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.datasets import make_classification
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

X, y = make_classification(20000, 20, n_informative=6, n_redundant=10, class_sep=0.8,
                           flip_y=0.0, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0)

raw = GaussianNB().fit(X_tr, y_tr)
cal = CalibratedClassifierCV(GaussianNB(), method="isotonic", cv=5).fit(X_tr, y_tr)
for name, m in (("raw", raw), ("isotonic", cal)):
    p = m.predict_proba(X_te)[:, 1]
    print(f"{name:9s} brier={brier_score_loss(y_te, p):.4f} logloss={log_loss(y_te, p):.4f}")

raw       brier=0.1139 logloss=0.4070
isotonic  brier=0.1078 logloss=0.3460


## 19.7 Feature Selection, Model Inspection, Saving Models and Custom Transformers

Once a model scores well, four practical questions follow. Do I need all these columns? Why did the model decide that? How do I ship it? And what if scikit-learn has no transformer for my cleaning step? This section answers all four.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.feature_selection import SelectFromModel, SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_classification(400, 30, n_informative=5, n_redundant=0, random_state=0)
lr = LogisticRegression(max_iter=1000)
variants = {
    "all 30 columns": make_pipeline(StandardScaler(), lr),
    "SelectKBest k=5": make_pipeline(StandardScaler(), SelectKBest(f_classif, k=5), lr),
    "L1 SelectFromModel": make_pipeline(
        StandardScaler(),
        SelectFromModel(LogisticRegression(l1_ratio=1, solver="liblinear", C=0.1)),
        lr),
}
for name, m in variants.items():
    print(f"{name:20s} {cross_val_score(m, X, y, cv=5).mean():.3f}")

fitted = variants["SelectKBest k=5"].fit(X, y)
print("kept columns:", fitted[1].get_support().nonzero()[0])

all 30 columns       0.865
SelectKBest k=5      0.885
L1 SelectFromModel   0.883
kept columns: [ 5  6 14 15 16]


In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

X, y = make_classification(1500, 5, n_informative=3, n_redundant=0, random_state=0, shuffle=False)
rng = np.random.default_rng(0)
X = np.c_[X, rng.normal(size=len(X))]                  # column 5: pure noise with many distinct values
names = ["f0", "f1", "f2", "f3", "f4", "noise"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0)

rf = RandomForestClassifier(300, min_samples_leaf=1, random_state=0).fit(X_tr, y_tr)
pi = permutation_importance(rf, X_te, y_te, n_repeats=20, random_state=0)
print(f"{'feature':8s} {'builtin':>8s} {'permutation':>12s}")
for i, n in enumerate(names):
    print(f"{n:8s} {rf.feature_importances_[i]:8.3f} {pi.importances_mean[i]:8.3f} +/- {pi.importances_std[i]:.3f}")

feature   builtin  permutation
f0          0.277    0.285 +/- 0.025
f1          0.209    0.146 +/- 0.018
f2          0.249    0.227 +/- 0.016
f3          0.087   -0.003 +/- 0.007
f4          0.085   -0.011 +/- 0.006
noise       0.092   -0.009 +/- 0.006


In [ ]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import partial_dependence

rng = np.random.default_rng(0)
X = rng.uniform(0, 10, size=(2000, 2))
y = 3 * np.sin(X[:, 0]) + 0.5 * X[:, 1] + rng.normal(scale=0.3, size=2000)
m = HistGradientBoostingRegressor(random_state=0).fit(X, y)

pd0 = partial_dependence(m, X, features=[0], grid_resolution=6, method="brute")
print("grid  :", pd0["grid_values"][0].round(2))
print("effect:", pd0["average"][0].round(2))
print("truth :", (3 * np.sin(pd0["grid_values"][0]) + 0.5 * X[:, 1].mean()).round(2))

grid  : [0.49 2.27 4.06 5.85 7.64 9.43]
effect: [3.89 4.77 0.05 1.29 5.36 2.42]
truth : [3.9  4.79 0.11 1.25 5.43 2.47]


In [ ]:
import os
import tempfile

import joblib
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X, y)

with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, "model.joblib")
    joblib.dump({"model": pipe, "sklearn": "1.9.1", "features": 30}, path, compress=3)
    print("file exists:", os.path.exists(path), " size under 20 KB:", os.path.getsize(path) < 20000)
    bundle = joblib.load(path)

same = np.array_equal(pipe.predict(X), bundle["model"].predict(X))
print("identical predictions after reload:", same)
print(bundle["sklearn"], bundle["features"])

file exists: True  size under 20 KB: True
identical predictions after reload: True
1.9.1 30


In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.utils.validation import check_is_fitted, validate_data


class Winsorizer(TransformerMixin, BaseEstimator):
    def __init__(self, lower=0.05, upper=0.95):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = validate_data(self, X, dtype=float)
        self.lo_ = np.quantile(X, self.lower, axis=0)
        self.hi_ = np.quantile(X, self.upper, axis=0)
        return self

    def transform(self, X):
        check_is_fitted(self)
        X = validate_data(self, X, dtype=float, reset=False)
        return np.clip(X, self.lo_, self.hi_)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features if input_features is not None
                          else self.feature_names_in_, dtype=object)


df = pd.DataFrame({"a": [1, 2, 3, 4, 100.0], "b": [5, 6, 7, 8, 9.0]})
w = Winsorizer(0.0, 0.8).set_output(transform="pandas")
print(w.fit_transform(df))
print(clone(w).get_params())

rng = np.random.default_rng(0)
X = rng.normal(size=(300, 1)); X[:5] = 40                 # 5 wild outliers
y = 2 * X[:, 0] + rng.normal(scale=0.2, size=300)
gs = GridSearchCV(make_pipeline(Winsorizer(), LinearRegression()),
                  {"winsorizer__upper": [0.9, 0.99]}, cv=3).fit(X, y)
print(gs.best_params_)

a    b
0   1.0  5.0
1   2.0  6.0
2   3.0  7.0
3   4.0  8.0
4  23.2  8.2
{'lower': 0.0, 'upper': 0.8}
{'winsorizer__upper': 0.99}


## 19.8 Recipes: How Do I ...?

Each recipe is a complete, runnable answer to a real task. Read the six big ones in order (they build skills), then use the quick ones as lookups.

In [ ]:
import joblib, os, tempfile
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

rng = np.random.default_rng(0)
n = 4000
df = pd.DataFrame({
    "tenure": rng.integers(1, 72, n),
    "monthly": rng.normal(65, 20, n).round(2),
    "calls": rng.poisson(2, n),
    "contract": rng.choice(["monthly", "yearly", "two-year"], n, p=[0.55, 0.25, 0.20]),
    "payment": rng.choice(["card", "bank", "cheque"], n),
})
logit = (-2.5 - 0.04 * df["tenure"] + 0.02 * df["monthly"] + 0.3 * df["calls"]
         + np.where(df["contract"] == "monthly", 1.0, 0.0)
         + 0.02 * df["tenure"] * (df["contract"] == "monthly"))      # an interaction
df["churn"] = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
df.loc[rng.random(n) < 0.08, "monthly"] = np.nan

X, y = df.drop(columns="churn"), df["churn"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y, random_state=0)

num, cat = ["tenure", "monthly", "calls"], ["contract", "payment"]
lin_prep = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
candidates = {
    "logistic": Pipeline([("prep", lin_prep), ("clf", LogisticRegression(max_iter=1000))]),
    "hist boosting": Pipeline([("prep", ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), cat)], remainder="passthrough")),
        ("clf", HistGradientBoostingClassifier(random_state=0))]),
}
cv = StratifiedKFold(5, shuffle=True, random_state=0)
for name, pipe in candidates.items():
    r = cross_validate(pipe, X_tr, y_tr, cv=cv, scoring=["roc_auc", "average_precision"])
    print(f"{name:14s} auc={r['test_roc_auc'].mean():.3f} ap={r['test_average_precision'].mean():.3f}")

final = candidates["logistic"].fit(X_tr, y_tr)
p = final.predict_proba(X_te)[:, 1]
print("TEST auc:", round(roc_auc_score(y_te, p), 3), " ap:", round(average_precision_score(y_te, p), 3))
with tempfile.TemporaryDirectory() as d:
    joblib.dump(final, os.path.join(d, "churn.joblib"))
    print("reloaded ok:", np.allclose(joblib.load(os.path.join(d, "churn.joblib")).predict_proba(X_te)[:, 1], p))

logistic       auc=0.752 ap=0.550
hist boosting  auc=0.709 ap=0.501
TEST auc: 0.751  ap: 0.537
reloaded ok: True


In [ ]:
import numpy as np
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(1)
n = 1500
area = rng.uniform(40, 250, n)
rooms = np.clip((area / 40 + rng.normal(0, 0.7, n)).round(), 1, 8)
age = rng.uniform(0, 60, n)
price = np.exp(10.5 + 0.009 * area + 0.05 * rooms - 0.006 * age + rng.normal(0, 0.15, n))
X = np.c_[area, rooms, age]

cv = KFold(5, shuffle=True, random_state=0)
models = {
    "mean baseline": DummyRegressor(),
    "ridge on price": make_pipeline(StandardScaler(), Ridge()),
    "ridge on log(price)": TransformedTargetRegressor(
        make_pipeline(StandardScaler(), Ridge()), func=np.log, inverse_func=np.exp),
    "hist boosting": HistGradientBoostingRegressor(random_state=0),
}
for name, m in models.items():
    r = cross_validate(m, X, price, cv=cv,
                       scoring=["neg_root_mean_squared_error", "neg_mean_absolute_error", "r2"])
    print(f"{name:20s} RMSE={-r['test_neg_root_mean_squared_error'].mean():9.0f} "
          f"MAE={-r['test_neg_mean_absolute_error'].mean():9.0f} R2={r['test_r2'].mean():.3f}")

mean baseline        RMSE=   106683 MAE=    87385 R2=-0.000
ridge on price       RMSE=    41475 MAE=    31182 R2=0.848
ridge on log(price)  RMSE=    29709 MAE=    19842 R2=0.922
hist boosting        RMSE=    32853 MAE=    21958 R2=0.904


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(0)
pools = {
    "billing": "invoice refund charge payment card overcharged receipt billing price".split(),
    "technical": "error crash login password app update install bug screen".split(),
    "shipping": "delivery package courier late tracking address parcel shipped lost".split(),
}
common = "please help my the is not working since yesterday order account thanks".split()
texts, labels = [], []
for lab, pool in pools.items():
    for _ in range(200):
        words = [rng.choice(pool) if rng.random() < 0.45 else rng.choice(common) for _ in range(9)]
        texts.append(" ".join(words)); labels.append(lab)
print(texts[0])

X_tr, X_te, y_tr, y_te = train_test_split(texts, labels, stratify=labels, random_state=0)
pipe = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                     LogisticRegression(max_iter=1000))
print("CV acc:", cross_val_score(pipe, X_tr, y_tr, cv=5).mean().round(3))
pipe.fit(X_tr, y_tr)
print(classification_report(y_te, pipe.predict(X_te), digits=3))

vec, clf = pipe[0], pipe[-1]
names = np.array(vec.get_feature_names_out())
for i, lab in enumerate(clf.classes_):
    print(lab, "->", names[np.argsort(clf.coef_[i])[-4:][::-1]].tolist())
print(pipe.predict(["my parcel never arrived and tracking is wrong"]))

working charge refund order thanks yesterday the billing working
CV acc: 0.998
              precision    recall  f1-score   support

     billing      0.980     1.000     0.990        50
    shipping      1.000     0.980     0.990        50
   technical      1.000     1.000     1.000        50

    accuracy                          0.993       150
   macro avg      0.993     0.993     0.993       150
weighted avg      0.993     0.993     0.993       150

billing -> ['price', 'payment', 'invoice', 'overcharged']
shipping -> ['courier', 'address', 'tracking', 'shipped']
technical -> ['screen', 'install', 'login', 'bug']
['shipping']


In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
parts = []
for recency, freq, money, size in [(10, 20, 2000, 150), (60, 6, 500, 250), (200, 2, 120, 200)]:
    parts.append(pd.DataFrame({
        "recency": rng.gamma(4, recency / 4, size),
        "frequency": rng.poisson(freq, size) + 1,
        "monetary": rng.lognormal(np.log(money), 0.35, size)}))
rfm = pd.concat(parts, ignore_index=True)

Z = StandardScaler().fit_transform(np.log1p(rfm))        # log tames skew, scaler equalises units
for k in range(2, 7):
    km = KMeans(k, n_init=10, random_state=0).fit(Z)
    print(k, "silhouette", round(silhouette_score(Z, km.labels_), 3), " inertia", round(km.inertia_, 1))

best = KMeans(3, n_init=10, random_state=0).fit(Z)
rfm["segment"] = best.labels_
print(rfm.groupby("segment").agg(n=("recency", "size"), recency=("recency", "mean"),
                                 frequency=("frequency", "mean"), monetary=("monetary", "mean")).round(0))

2 silhouette 0.561  inertia 658.6
3 silhouette 0.567  inertia 274.2
4 silhouette 0.482  inertia 236.2
5 silhouette 0.388  inertia 202.1
6 silhouette 0.385  inertia 183.3
           n  recency  frequency  monetary
segment                                   
0        247     58.0        7.0     551.0
1        202    202.0        3.0     130.0
2        151      9.0       21.0    2096.0


In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from sklearn.neighbors import LocalOutlierFactor

rng = np.random.default_rng(0)
normal = np.c_[rng.lognormal(3.5, 0.5, 2000), rng.normal(14, 3, 2000)]      # amount, hour
odd = np.c_[rng.lognormal(6.0, 0.4, 40), rng.normal(3, 1, 40)]              # big, at 3 am
X = np.vstack([normal, odd])
is_anomaly = np.r_[np.zeros(2000, int), np.ones(40, int)]

iso = IsolationForest(n_estimators=200, contamination=0.02, random_state=0).fit(X)
pred = (iso.predict(X) == -1).astype(int)            # predict gives -1 for outliers
score = -iso.score_samples(X)                         # higher = more anomalous
print("isolation forest: AUC", round(roc_auc_score(is_anomaly, score), 3),
      " precision", round(precision_score(is_anomaly, pred), 3),
      " recall", round(recall_score(is_anomaly, pred), 3))

lof = LocalOutlierFactor(n_neighbors=30, contamination=0.02)
pred2 = (lof.fit_predict(X) == -1).astype(int)
print("local outlier factor: precision", round(precision_score(is_anomaly, pred2), 3),
      " recall", round(recall_score(is_anomaly, pred2), 3))
top = np.argsort(score)[-10:]
print("top-10 flagged rows that are real anomalies:", int(is_anomaly[top].sum()))

isolation forest: AUC 1.0  precision 0.976  recall 1.0
local outlier factor: precision 0.463  recall 0.475
top-10 flagged rows that are real anomalies: 10


In [ ]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

leaky, honest = [], []
for seed in range(10):
    r = np.random.RandomState(seed)
    X = r.normal(size=(60, 2000))
    y = r.randint(0, 2, 60)

    X_sel = SelectKBest(f_classif, k=20).fit_transform(X, y)         # selection sees ALL labels
    leaky.append(cross_val_score(LogisticRegression(), X_sel, y, cv=5).mean())

    pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression())
    honest.append(cross_val_score(pipe, X, y, cv=5).mean())          # selection redone per fold

print("leaky  mean accuracy:", round(float(np.mean(leaky)), 3))
print("honest mean accuracy:", round(float(np.mean(honest)), 3))

leaky  mean accuracy: 0.917
honest mean accuracy: 0.53


In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=0)   # 20 identical folds for both
a = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X, y, cv=cv)
b = cross_val_score(RandomForestClassifier(200, random_state=0), X, y, cv=cv)
diff = a - b
print("means:", a.mean().round(3), b.mean().round(3))
print("paired diff mean:", diff.mean().round(4), " std:", diff.std(ddof=1).round(4),
      " logistic wins in", int((diff > 0).sum()), "of 20 folds")

means: 0.975 0.963
paired diff mean: 0.0123  std: 0.0188  logistic wins in 13 of 20 folds


In [ ]:
from sklearn.datasets import load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_wine(return_X_y=True)
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
oof = cross_val_predict(pipe, X, y, cv=5)
print(confusion_matrix(y, oof))
print("oof accuracy:", round((oof == y).mean(), 3))

[[58  1  0]
 [ 0 69  2]
 [ 0  0 48]]
oof accuracy: 0.983


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = load_breast_cancer(return_X_y=True)
base = [("svm", make_pipeline(StandardScaler(), SVC())),
        ("rf", RandomForestClassifier(200, random_state=0)),
        ("hgb", HistGradientBoostingClassifier(random_state=0))]
stack = StackingClassifier(base, final_estimator=LogisticRegression(max_iter=1000), cv=5)
print("stack:", cross_val_score(stack, X, y, cv=5).mean().round(3))
print("svm  :", cross_val_score(base[0][1], X, y, cv=5).mean().round(3))

stack: 0.977
svm  : 0.974


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.DataFrame({"age": [25, 40, 31], "city": ["Oslo", "Rome", "Oslo"]})
ct = ColumnTransformer([("num", StandardScaler(), ["age"]),
                        ("cat", OneHotEncoder(sparse_output=False), ["city"])],
                       verbose_feature_names_out=False).set_output(transform="pandas")
out = ct.fit_transform(df)
print(out.round(2))
print(type(out).__name__, list(out.columns))

age  city_Oslo  city_Rome
0 -1.14        1.0        0.0
1  1.30        0.0        1.0
2 -0.16        1.0        0.0
DataFrame ['age', 'city_Oslo', 'city_Rome']


## 19.9 Performance and Pitfalls

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

train = np.array([[10.0], [20.0], [30.0]])
test = np.array([[100.0], [110.0], [120.0]])      # the world has shifted upward

right = StandardScaler().fit(train).transform(test)
wrong = StandardScaler().fit_transform(test)
print("right:", right.ravel().round(2))
print("wrong:", wrong.ravel().round(2))

right: [ 9.8  11.02 12.25]
wrong: [-1.22  0.    1.22]


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X = pd.DataFrame({"age": [20, 30, 40, 50], "income": [10, 20, 30, 40]})
y = [0, 0, 1, 1]
bare = LogisticRegression().fit(X, y)
safe = make_pipeline(ColumnTransformer([("s", StandardScaler(), ["age", "income"])]),
                     LogisticRegression()).fit(X, y)

for label, bad in (("renamed", X.rename(columns={"income": "salary"})),
                   ("reordered", X[["income", "age"]])):
    try:
        bare.predict(bad)
    except ValueError as e:
        print(label, "-> bare model:", str(e).splitlines()[1])
print("reordered -> pipeline with ColumnTransformer:", safe.predict(X[["income", "age"]]))

renamed -> bare model: Feature names unseen at fit time:
reordered -> bare model: Feature names must be in the same order as they were in fit.
reordered -> pipeline with ColumnTransformer: [0 0 1 1]


In [ ]:
import numpy as np
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_val_score

X = np.arange(20).reshape(-1, 1)
y = np.arange(20) * 1.0
scores = cross_val_score(DummyRegressor(), X, y, cv=4, scoring="neg_mean_squared_error")
print(scores.round(1))
print("report as MSE:", round(-scores.mean(), 1))

[-102.   -13.1  -13.1 -102. ]
report as MSE: 57.6


## 19.10 Cross-Library Integration

Scikit-learn rarely works alone. This section shows the handshakes with the libraries around it. The first three examples are run; the rest are short notes.

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

rng = np.random.default_rng(0)
con = sqlite3.connect(":memory:")
n = 800
pd.DataFrame({"plan": rng.choice(["basic", "plus", "pro"], n),
              "age": rng.integers(18, 70, n),
              "visits": rng.poisson(5, n)}).assign(
    bought=lambda d: ((d["visits"] > 5) ^ (rng.random(n) < 0.15)).astype(int)
).to_sql("customers", con, index=False)

df = pd.read_sql("SELECT plan, age, visits, bought FROM customers", con)
print(df["plan"].dtype, df["age"].dtype)

X, y = df.drop(columns="bought"), df["bought"]
prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"),
                           make_column_selector(dtype_include=object))],
                         remainder="passthrough")
pipe = make_pipeline(prep, HistGradientBoostingClassifier(random_state=0))
print("cv accuracy:", cross_val_score(pipe, X, y, cv=5).mean().round(3))

str int64
cv accuracy: 0.798


In [ ]:
import os
import tempfile
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data = load_wine()
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
pred = cross_val_predict(pipe, data.data, data.target, cv=5)
cm = confusion_matrix(data.target, pred)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
ConfusionMatrixDisplay(cm, display_labels=data.target_names).plot(ax=axes[0], colorbar=False)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[1],
            xticklabels=data.target_names, yticklabels=data.target_names)
with tempfile.TemporaryDirectory() as d:
    fig.savefig(os.path.join(d, "cm.png"), dpi=80)
    print("saved:", os.path.getsize(os.path.join(d, "cm.png")) > 1000)
print(cm.diagonal().sum(), "of", cm.sum(), "correct")

saved: True
175 of 178 correct


In [ ]:
import json
import threading
from http.server import BaseHTTPRequestHandler, HTTPServer

import requests
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression

iris = load_iris()
model = LogisticRegression(max_iter=300).fit(iris.data, iris.target)

class Handler(BaseHTTPRequestHandler):
    def do_POST(self):
        body = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
        pred = model.predict([body["features"]])[0]
        out = json.dumps({"species": str(iris.target_names[pred])}).encode()
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(out)))
        self.end_headers()
        self.wfile.write(out)
    def log_message(self, *args):
        pass

server = HTTPServer(("127.0.0.1", 0), Handler)       # port 0 = pick any free port
threading.Thread(target=server.serve_forever, daemon=True).start()
url = f"http://127.0.0.1:{server.server_port}/predict"

for row in ([5.0, 3.4, 1.5, 0.2], [6.7, 3.0, 5.2, 2.3]):
    print(requests.post(url, json={"features": row}, timeout=5).json())
server.shutdown()

{'species': 'setosa'}
{'species': 'virginica'}


## 19.11 How to Solve Any scikit-learn Problem: A Playbook

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
n = 3000
df = pd.DataFrame({"tenure": rng.integers(1, 60, n), "calls": rng.poisson(2, n)})
p = 1 / (1 + np.exp(-(0.4 * df["calls"] - 0.03 * df["tenure"] - 0.5)))
df["churn"] = (rng.random(n) < p).astype(int)
df["refund_issued"] = df["churn"] * (rng.random(n) < 0.9)       # consequence of the label

def run(cols):
    X_tr, X_te, y_tr, y_te = train_test_split(df[cols], df["churn"], random_state=0)
    m = HistGradientBoostingClassifier(random_state=0).fit(X_tr, y_tr)
    return m, X_te, y_te, roc_auc_score(y_te, m.predict_proba(X_te)[:, 1])

m, X_te, y_te, auc = run(["tenure", "calls", "refund_issued"])
imp = permutation_importance(m, X_te, y_te, n_repeats=5, random_state=0, scoring="roc_auc")
print("with leak AUC:", round(auc, 3), {c: round(float(v), 3) for c, v in zip(X_te.columns, imp.importances_mean)})
print("fixed AUC    :", round(run(["tenure", "calls"])[3], 3))

with leak AUC: 0.945 {'tenure': 0.002, 'calls': -0.002, 'refund_issued': 0.398}
fixed AUC    : 0.646


In [ ]:
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X, y = make_classification(1500, 20, n_informative=5, flip_y=0.15, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0)
for leaf in (1, 10, 30):
    m = RandomForestClassifier(200, min_samples_leaf=leaf, random_state=0).fit(X_tr, y_tr)
    print(f"min_samples_leaf={leaf:2d} train={m.score(X_tr, y_tr):.3f} test={m.score(X_te, y_te):.3f}")

min_samples_leaf= 1 train=1.000 test=0.829
min_samples_leaf=10 train=0.891 test=0.832
min_samples_leaf=30 train=0.832 test=0.824


In [ ]:
from sklearn.datasets import make_moons
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

X, y = make_moons(800, noise=0.25, random_state=0)
options = {
    "logistic": LogisticRegression(),
    "logistic + degree-3 polynomial": make_pipeline(PolynomialFeatures(3), StandardScaler(), LogisticRegression(max_iter=1000)),
    "hist boosting": HistGradientBoostingClassifier(random_state=0),
}
for name, m in options.items():
    print(f"{name:32s} {cross_val_score(m, X, y, cv=5).mean():.3f}")

logistic                         0.859
logistic + degree-3 polynomial   0.909
hist boosting                    0.938


In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import TunedThresholdClassifierCV, train_test_split

X, y = make_classification(8000, 12, n_informative=4, weights=[0.95], flip_y=0.02, random_state=2)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y, random_state=2)

plain = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
tuned = TunedThresholdClassifierCV(LogisticRegression(max_iter=1000), scoring="f1", cv=5).fit(X_tr, y_tr)
print("default 0.5 threshold F1:", round(f1_score(y_te, plain.predict(X_te)), 3))
print("tuned threshold      F1:", round(f1_score(y_te, tuned.predict(X_te)), 3),
      " threshold:", round(float(tuned.best_threshold_), 3))

default 0.5 threshold F1: 0.481
tuned threshold      F1: 0.53  threshold: 0.22


In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)
X, y = X[:200], y[:200]                       # a small sample makes estimates noisy
m = DecisionTreeClassifier(max_depth=3, random_state=0)
for seed in (0, 1, 2):
    s = cross_val_score(m, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=seed)).mean()
    print("single 5-fold, seed", seed, round(s, 4))
rep = cross_val_score(m, X, y, cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0))
print("repeated 5x10 mean:", rep.mean().round(4), " std error:", (rep.std() / np.sqrt(len(rep))).round(4))

single 5-fold, seed 0 0.895
single 5-fold, seed 1 0.89
single 5-fold, seed 2 0.905
repeated 5x10 mean: 0.9025  std error: 0.0052


In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder

train = pd.DataFrame({"plan": ["a", "b", "a", "b", "a", "b"], "usage": [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]})
y = [0, 0, 0, 1, 1, 1]
prep = ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore"), ["plan"]),
                          ("n", SimpleImputer(strategy="median"), ["usage"])])
pipe = Pipeline([("prep", prep), ("clf", LogisticRegression())]).fit(train, y)

def check_input(df, expected=("plan", "usage")):
    missing = [c for c in expected if c not in df.columns]
    if missing:
        raise ValueError(f"missing columns: {missing}")
    return df[list(expected)]

live = pd.DataFrame({"usage": [np.nan, 5.5], "plan": ["zzz", "b"], "extra": [1, 2]})
print(pipe.predict(check_input(live)))
try:
    check_input(live.drop(columns="plan"))
except ValueError as e:
    print(e)

[1 1]
missing columns: ['plan']


In [ ]:
import time
from sklearn.datasets import make_classification
from sklearn.linear_model import SGDClassifier
from sklearn.svm import LinearSVC, SVC

X, y = make_classification(15000, 30, n_informative=10, random_state=0)

def seconds(model):
    start = time.perf_counter(); model.fit(X, y); return time.perf_counter() - start

t_rbf = seconds(SVC())
t_lin = seconds(LinearSVC(dual=False))
t_sgd = seconds(SGDClassifier(random_state=0))
print("LinearSVC at least 5x faster than RBF SVC:", t_lin * 5 < t_rbf)
print("SGDClassifier at least 5x faster than RBF SVC:", t_sgd * 5 < t_rbf)

LinearSVC at least 5x faster than RBF SVC: True
SGDClassifier at least 5x faster than RBF SVC: True


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

data = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target, random_state=0)
black_box = HistGradientBoostingClassifier(random_state=0).fit(X_tr, y_tr)

surrogate = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_tr, black_box.predict(X_tr))
fidelity = (surrogate.predict(X_te) == black_box.predict(X_te)).mean()
print("black box accuracy:", round(black_box.score(X_te, y_te), 3), " surrogate fidelity:", round(fidelity, 3))
print(export_text(surrogate, feature_names=list(data.feature_names)))

black box accuracy: 0.986  surrogate fidelity: 0.937
|--- mean concave points <= 0.05
|   |--- worst area <= 952.90
|   |   |--- class: 1
|   |--- worst area >  952.90
|   |   |--- class: 0
|--- mean concave points >  0.05
|   |--- worst area <= 785.80
|   |   |--- class: 1
|   |--- worst area >  785.80
|   |   |--- class: 0


In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import permutation_test_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X, y = load_breast_cancer(return_X_y=True)
print("shape", X.shape, " NaN:", int(np.isnan(X).sum()), " class counts:", np.bincount(y))

score, perm_scores, pvalue = permutation_test_score(
    make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X, y,
    cv=5, n_permutations=20, random_state=0)
print("real score:", round(score, 3), " shuffled-label mean:", perm_scores.mean().round(3),
      " p-value:", round(pvalue, 3))

tiny = RandomForestClassifier(random_state=0).fit(X[:20], y[:20])
print("can it memorise 20 rows?", tiny.score(X[:20], y[:20]))

shape (569, 30)  NaN: 0  class counts: [212 357]
real score: 0.981  shuffled-label mean: 0.602  p-value: 0.048
can it memorise 20 rows? 1.0


## 19.12 Going Deeper: Internals, Memory, Speed and What Interviewers Probe

This section is for when you want to know *why* the library behaves as it does. Skip it on a first read.

In [ ]:
from sklearn.base import BaseEstimator, clone
from sklearn.linear_model import LogisticRegression


class Bad(BaseEstimator):
    def __init__(self, scale=1.0):
        self.scale = scale * 2          # forbidden: modifies the argument in the constructor


class Good(BaseEstimator):
    def __init__(self, scale=1.0):
        self.scale = scale


print(clone(Good(3.0)).scale)
try:
    clone(Bad(3.0))
except RuntimeError as e:
    print("clone failed:", str(e).splitlines()[0][:80])

tags = LogisticRegression().__sklearn_tags__()
print("sparse input:", tags.input_tags.sparse, " allows NaN:", tags.input_tags.allow_nan)

3.0
clone failed: Cannot clone object Bad(scale=6.0), as the constructor either does not set or mo
sparse input: True  allows NaN: False


In [ ]:
import numpy as np
from scipy import sparse

dense64 = np.zeros((10000, 1000))
dense32 = dense64.astype(np.float32)
rng = np.random.default_rng(0)
mask = rng.random((10000, 1000)) < 0.01                 # 1% non-zero
csr = sparse.csr_matrix(np.where(mask, 1.0, 0.0))

print("float64 MB:", dense64.nbytes / 1e6)
print("float32 MB:", dense32.nbytes / 1e6)
print("csr MB    :", round((csr.data.nbytes + csr.indices.nbytes + csr.indptr.nbytes) / 1e6, 2))

float64 MB: 80.0
float32 MB: 40.0
csr MB    : 1.24


In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import learning_curve

X, y = make_classification(3000, 20, n_informative=6, flip_y=0.1, random_state=0)
sizes, tr, va = learning_curve(LogisticRegression(max_iter=1000), X, y,
                               train_sizes=[0.05, 0.1, 0.25, 0.5, 1.0], cv=5, random_state=0)
for n, a, b in zip(sizes, tr.mean(axis=1), va.mean(axis=1)):
    print(f"n={n:5d} train={a:.3f} validation={b:.3f} gap={a - b:.3f}")

n=  120 train=0.868 validation=0.757 gap=0.111
n=  240 train=0.839 validation=0.789 gap=0.050
n=  600 train=0.816 validation=0.792 gap=0.023
n= 1200 train=0.812 validation=0.801 gap=0.011
n= 2400 train=0.810 validation=0.807 gap=0.003


## Chapter recap

## Practice set

Twenty-three problems, easy to expert. Try each before reading the answer. Where a problem shows code, the printed output is given in the answer, and it was produced by running the code.

In [ ]:
# needs a full environment: pip install sklearn
from sklearn.preprocessing import OneHotEncoder

enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit([["a"], ["b"]])
print(enc.transform([["a"], ["c"]]))

In [ ]:
# needs a full environment: pip install numpy sklearn
import numpy as np
from sklearn.preprocessing import StandardScaler

sc = StandardScaler().fit(np.array([[1.0], [2.0], [3.0]]))
print(sc.mean_, sc.scale_.round(4))
print(sc.transform(np.array([[3.0], [4.0], [10.0]])).ravel().round(3))

In [ ]:
# needs a full environment: pip install sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

X_scaled = StandardScaler().fit_transform(X)
print(cross_val_score(LogisticRegression(), X_scaled, y, cv=5).mean())

In [ ]:
# needs a full environment: pip install sklearn
from sklearn.metrics import confusion_matrix

print(confusion_matrix([0, 1, 1, 1], [0, 0, 1, 1]))

In [ ]:
# needs a full environment: pip install numpy sklearn
import numpy as np
from sklearn.linear_model import LogisticRegression

a = LogisticRegression()
b = a
b.set_params(C=5)
print(a.C)

from sklearn.base import clone
c = clone(a)
c.set_params(C=9)
print(a.C, c.C)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.DataFrame({
    "age": [25, 32, np.nan, 41, 29, 55, 38, np.nan, 47, 23, 36, 60],
    "income": [30, 45, 52, np.nan, 38, 80, 61, 58, 72, 28, 49, 90],
    "city": ["A", "B", "A", "C", "B", "C", "A", "B", "C", "A", "B", "C"],
})
y = [0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1]

prep = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), ["age", "income"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["city"])])
pipe = make_pipeline(prep, LogisticRegression())
print(cross_val_score(pipe, df, y, cv=3).round(3))

[1.   0.75 1.  ]


In [ ]:
# needs a full environment: pip install pandas sklearn
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = pd.DataFrame({"n": [1.0, 2.0, 3.0, 4.0], "c": ["x", "y", "z", "x"]})
y = [0, 1, 0, 1]
pipe = make_pipeline(
    ColumnTransformer([("s", StandardScaler(), ["n"]), ("o", OneHotEncoder(), ["c"])]),
    LogisticRegression()).fit(X, y)
print(pipe[:-1].transform(X).shape, pipe[-1].coef_.shape)
print(pipe.steps[0][0])

In [ ]:
# needs a full environment: pip install sklearn
from sklearn.metrics import roc_auc_score

print(roc_auc_score([0, 0, 1, 1, 1], [0.1, 0.6, 0.4, 0.8, 0.7]))

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted


class TopK(TransformerMixin, BaseEstimator):
    def __init__(self, k=2):
        self.k = k

    def fit(self, X, y=None):
        values = np.asarray(X).ravel()
        self.keep_ = [v for v, _ in Counter(values).most_common(self.k)]
        return self

    def transform(self, X):
        check_is_fitted(self)
        values = np.asarray(X).ravel()
        return np.array([[v if v in self.keep_ else "other"] for v in values], dtype=object)


train = pd.DataFrame({"city": ["Oslo", "Oslo", "Rome", "Rome", "Rome", "Paris", "Lima"]})
t = TopK(k=2).fit(train[["city"]])
print(t.keep_)
print(t.transform(pd.DataFrame({"city": ["Rome", "Paris", "Berlin"]})).ravel())

['Rome', 'Oslo']
['Rome' 'other' 'other']


In [ ]:
# needs a full environment: pip install numpy sklearn
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
x = rng.normal(size=(1500, 1))
y = (x[:, 0] + rng.normal(scale=0.5, size=1500) > 0).astype(int)
X_single = np.c_[x, rng.normal(size=1500)]                    # [signal, noise]
X_dup = np.c_[x, x + rng.normal(scale=0.01, size=(1500, 1)), rng.normal(size=1500)]  # signal twice

for name, X in (("one copy", X_single), ("two copies", X_dup)):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0)
    m = RandomForestClassifier(200, random_state=0).fit(X_tr, y_tr)
    pi = permutation_importance(m, X_te, y_te, n_repeats=10, random_state=0)
    print(name, pi.importances_mean.round(3))

In [ ]:
# needs a full environment: pip install sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

X, y = make_classification(300, 8, random_state=0)
gs = GridSearchCV(LogisticRegression(max_iter=500), {"C": [0.01, 1.0]}, cv=3, refit=True).fit(X, y)
print([int(r) for r in sorted(gs.cv_results_["rank_test_score"])], hasattr(gs, "best_estimator_"), gs.n_features_in_)

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/python/ch19.html